# Stage 3 — Create LangChain Documents

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Convert each row of `train_prepared.csv` into a LangChain `Document` object — `page_content` (the searchable text) + `metadata` (ID, title, topics).

**Before running:** upload `train_prepared.csv` (created at the end of Stage 2) to this Colab session.

## Cell 1 — Install `langchain-core`

We only need the lightweight `langchain-core` package here, since all we're using is the `Document` class. We don't need the full `langchain` package until later stages (chains, retrievers).

In [1]:
!pip install -q langchain-core

## Cell 2 — Load the prepared dataset

Loads `train_prepared.csv` (the output of Stage 2) — already cleaned, with the combined `text` field and readable `topics` field ready to use.

In [2]:
import pandas as pd

df = pd.read_csv("train_prepared.csv")

print("Shape:", df.shape)
df.head(3)

Shape: (20971, 5)


,ID,TITLE,ABSTRACT,text,topics
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,Reconstructing Subject-Specific Effect Maps\n\...,Computer Science
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,Rotation Invariance Neural Network\n\n Rotati...,Computer Science
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,Spherical polyharmonics and Poisson kernels fo...,Mathematics


## Cell 3 — What goes into `page_content` vs `metadata`

- **`page_content`** = the `text` column (title + abstract) — this is what gets embedded and searched in Stage 5/6.
- **`metadata`** = `id`, `title`, `topics` — not searched directly, but carried along with each document so we can later show *which paper* an answer came from, without needing to search that information.

This split matters: if we stuffed the ID or topics into `page_content` too, they'd get embedded and could subtly distort similarity search (e.g. the string "Computer Science" appearing in the embedded text of thousands of papers). Keeping them in `metadata` avoids that.

## Cell 4 — Build the list of `Document` objects

For every row, we create one `Document`: `page_content` is the combined text, `metadata` is a small dictionary of the paper's ID, title, and topics.

In [3]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=row["text"],
        metadata={
            "id": row["ID"],
            "title": row["TITLE"],
            "topics": row["topics"],
        }
    )
    documents.append(doc)

print("Total documents created:", len(documents))

Total documents created: 20971


## Cell 5 — Inspect a few example documents

Print the first 2 documents in full, so we can see exactly what `page_content` and `metadata` look like on a real `Document` object.

In [4]:
for doc in documents[:2]:
    print("=" * 60)
    print("PAGE CONTENT:\n", doc.page_content)
    print("\nMETADATA:", doc.metadata)

PAGE CONTENT:
 Reconstructing Subject-Specific Effect Maps

  Predictive models allow subject-specific inference when analyzing disease
related alterations in neuroimaging data. Given a subject's data, inference can
be made at two levels: global, i.e. identifiying condition presence for the
subject, and local, i.e. detecting condition effect on each individual
measurement extracted from the subject's data. While global inference is widely
used, local inference, which can be used to form subject-specific effect maps,
is rarely used because existing models often yield noisy detections composed of
dispersed isolated islands. In this article, we propose a reconstruction
method, named RSM, to improve subject-specific detections of predictive
modeling approaches and in particular, binary classifiers. RSM specifically
aims to reduce noise due to sampling error associated with using a finite
sample of examples to train classifiers. The proposed method is a wrapper-type
algorithm that can be us

## Cell 6 — Sanity checks

Confirm: the number of documents matches the number of dataset rows, and every document has all 3 expected metadata keys.

In [5]:
assert len(documents) == len(df), "Mismatch between number of documents and dataset rows!"

expected_keys = {"id", "title", "topics"}
missing = [i for i, d in enumerate(documents) if set(d.metadata.keys()) != expected_keys]

print("Document count matches dataset rows:", len(documents) == len(df))
print("Documents with missing/extra metadata keys:", len(missing))

Document count matches dataset rows: True
Documents with missing/extra metadata keys: 0


## What to check after running this notebook

- **Cell 4:** total document count should be 20,971 (matching Stage 2's cleaned dataset).
- **Cell 5:** confirm `page_content` looks like title + abstract, and `metadata` shows `id`, `title`, `topics` correctly.
- **Cell 6:** both checks should pass — document count matches, and 0 documents have missing/extra metadata keys.

Note: `documents` is an in-memory Python list in this notebook session — it isn't saved to a file, because a `Document` object isn't naturally CSV-shaped. In Stage 4 (Text Splitting) and beyond, we'll simply re-run this same document-creation step at the start of each notebook (it only takes a couple of seconds), building on `train_prepared.csv` each time.

Paste back the total document count from Cell 4 and the Cell 6 sanity-check results — then we'll move to Stage 4 (Text Splitting).